# 04 – Availability and rental model

## Question
- **Question:** Is Airbnb in these cities still a short-stay (tourist) product, or has a part of it turned into mid- and long-term rental (30+ nights)? How does the rental model relate to how available a listing is during the year?
- **Hypothesis (before looking at the data):** long-term listings (`minimum_nights >= 30`) are more common in Los Angeles than in Barcelona.
- **Variables:** `minimum_nights`, `availability_365`, `room_type`, `neighbourhood_cleansed`.
- **Methods:** missing-value check, frequency table of `minimum_nights`, share of 30+ listings per city, histograms and boxplots of `availability_365`, statistical tests (chi-square, Mann-Whitney), choropleth map of the 30+ share per neighbourhood, replication on pairs 2 and 3 with the same code.

## 1. Load the data

Load only the columns this study needs (19 of 90) for the cities of pair 1 (Barcelona, Los Angeles), the discovery pair. The replication later runs the same cells for pairs 2 and 3. A readable `city_name` column is added for plots, and the last line shows the number of listings per city.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.cities import CITIES, cities_in_pair
from src.load import load_all

FIG_DIR = Path("../figures")  # PNGs for the report
FIG_DIR.mkdir(exist_ok=True)

cols = [
    "id", "host_id", "room_type", "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "latitude", "longitude", "price",
    "minimum_nights", "maximum_nights", "minimum_minimum_nights", "maximum_minimum_nights",
    "has_availability", "availability_30", "availability_90", "availability_365",
    "instant_bookable", "number_of_reviews_ltm", "estimated_occupancy_l365d",
    "license", "calculated_host_listings_count",
]

# Discovery on pair 1 first; replication later runs the same cells with all cities
df = load_all(cities_in_pair(1), usecols=cols)
df["city_name"] = df["city"].map({k: c.name for k, c in CITIES.items()})
df.groupby("city_name").size()

## 2. Missing values

Before any analysis, check which of the loaded columns have missing values, as a percentage per city. This tells us which variables can be used at all and whether a missing share differs between cities (which would itself need an explanation).

**Result:** `instant_bookable` is empty (100% missing) in both cities, and in all six cities of the project, so it is dropped from the study. The core variables of this study (`minimum_nights`, `availability_365`, `room_type`) are complete.

In [ ]:
# Share of missing values (%) per column and city; only columns with any missing values are shown
missing = df[cols].isna().groupby(df["city_name"]).mean().mul(100).round(1).T
missing[missing.max(axis=1) > 0]

## 3. Are listings without a price random?

`price` is missing for 12.7% of listings. Dropping these rows is only safe if they are a random subset. To check, compare listings with and without a price on availability and on recent reviews (a demand proxy).

**Result:** listings without a price have a median `availability_365` of 0, and about 72% of them have no free night at all, against 0% of listings with a price. They also have a median of 0 reviews in the last 12 months. The missing price is therefore not random: it marks listings that cannot be booked at the moment. Our interpretation is that these are inactive ("ghost") listings that are still visible on the platform. **Rows without a price are not dropped:** price is not a variable in this study, and dropping them would remove exactly the group that matters for an availability study (selection bias). Analyses that need the price drop missing values locally.

In [ ]:
# Compare listings with and without a price: availability and recent demand
df["price_missing"] = df["price"].isna()
df.groupby(["city_name", "price_missing"]).agg(
    listings=("id", "size"),
    median_availability_365=("availability_365", "median"),
    pct_availability_0=("availability_365", lambda s: (s == 0).mean() * 100),
    median_reviews_ltm=("number_of_reviews_ltm", "median"),
).round(1)

## 4. Inactive ("ghost") listings

A listing with `availability_365 == 0` is either booked out for the whole year or blocked by the host. Reviews separate the two cases: a listing that is booked out should keep receiving reviews. We define an **inactive listing** as one with zero availability **and** no review in the last 12 months, and keep it in the data with a flag (`is_inactive`) instead of dropping it. Later analyses of availability can then be run with and without these listings.

**Result:** about 9% of listings in both cities have zero availability, and most of them (86% in Barcelona, 82% in Los Angeles) had no review in the last 12 months. Inactive listings make up 7.8% of Barcelona and 7.6% of Los Angeles. The few zero-availability listings with recent reviews (196 and 728) may be booked out or blocked only recently, so they are not flagged.

In [ ]:
# Flag "ghost" listings: no free night in the next 365 days and no review in the last 12 months
df["is_inactive"] = (df["availability_365"] == 0) & (df["number_of_reviews_ltm"] == 0)

# Of the listings with zero availability, how many still got reviews (booked out) and how many did not (ghost)?
zero_avail = df[df["availability_365"] == 0]
print(pd.crosstab(zero_avail["city_name"], zero_avail["number_of_reviews_ltm"] > 0)
        .rename(columns={False: "no reviews in 12 months", True: "has reviews"}))

# Share of inactive listings (%) per city
df.groupby("city_name")["is_inactive"].mean().mul(100).round(1)

## 5. Minimum stay: the most common values

`minimum_nights` is the shortest stay a host accepts. It is the clearest signal of the rental model a host has chosen: 1–2 nights is a tourist product, 30+ nights is closer to a monthly rental. The table lists the 8 most common values per city (as % of the city's listings). The chart shows the whole distribution up to 40 nights, one panel per city.

**Result:** the distribution is bimodal in both cities: one peak at 1–2 nights and a second peak at about one month, with almost nothing between 7 and 29 nights. The position of the second peak differs. In Barcelona it sits at **31 and 32 nights** (17.3% each), while exactly 30 nights is only 1.0%. In Los Angeles it sits at **30 nights** (35.7%, the most common value of all), followed by 31 nights (9.6%).

**Interpretation:** the second peak sits at each city's legal boundary between a tourist rental and an ordinary rental.
- **Barcelona:** in Catalonia, a tourist-use dwelling (*habitatge d'ús turístic*, HUT) is a dwelling let for stays of **31 days or less**, and it needs a tourist licence (HUTB number). Barcelona has issued no new licences since 2014 and plans to phase out the existing ones by 2028. A stay longer than 31 days is a seasonal rental (*lloguer de temporada*) under the national Urban Leases Law (LAU) and needs no tourist licence.
- **Los Angeles:** the Home-Sharing Ordinance (Ordinance 185931, in force since 2019) defines a short-term rental as transient use of **30 consecutive days or less**. Short-term renting requires a city registration, is allowed only in the host's primary residence, and is capped at 120 days a year. A host who rents a second home, or does not want to register, can only offer longer stays.
- In both cities the peak is split over two values (31/32 and 30/31). This is consistent with hosts being unsure whether the limit counts nights or calendar days and adding one night to stay safely outside it. This part is a hypothesis: the data shows where hosts put the minimum, not why. Section 7 tests the Barcelona explanation against the licence field.

Sources (accessed 2026-10-25): Generalitat de Catalunya, *Declaració responsable d'habitatges d'ús turístic* (https://www.accio.gencat.cat/en/serveis/convocatories-dajuts/llistat-ajuts/tramits/Declaracio-responsable-dhabitatges-dus-turistic-a-lajuntament); City of Los Angeles, Home-Sharing Ordinance (https://planning.lacity.gov/ordinances/docs/HomeSharing/adopted/Final%20Ordinance.pdf) and Council File 14-1635-S7 (https://cityclerk.lacity.org/onlinedocs/2014/14-1635-S7_misc_08-27-2020.pdf); licence freeze and 2028 phase-out in Barcelona as summarised by secondary sources, e.g. https://www.hostaway.com/blog/airbnb-rules-barcelona/.

In [ ]:
# The 8 most common minimum_nights values per city, as % of the city's listings
(
    df.groupby("city_name")["minimum_nights"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .groupby("city_name").head(8)
    .rename("pct_of_listings")
    .reset_index()
    .astype({"minimum_nights": "Int64"})  # stored as float because of missing values; nights are whole numbers
)

In [ ]:
# Distribution of minimum_nights from 1 to 40 nights, one panel per city (% of all listings of the city)
city_names = df["city_name"].unique()
fig, axes = plt.subplots(1, len(city_names), figsize=(12, 4), sharey=True)
for ax, city in zip(axes, city_names):
    nights = df.loc[df["city_name"] == city, "minimum_nights"]
    pct = nights[nights <= 40].value_counts().sort_index() / len(nights) * 100
    ax.bar(pct.index, pct.values, color="#2a78d6", width=0.8)
    ax.set_title(city)
    ax.set_xlabel("Minimum nights")
    ax.set_xticks([1, 7, 14, 21, 30, 40])
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("% of listings")
fig.suptitle("Minimum stay required by hosts (values above 40 nights not shown)")
plt.tight_layout()
plt.savefig(FIG_DIR / "study04_min_nights_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Rental model per city (prediction check)

Group listings into three rental models by `minimum_nights`: **short stay** (1–6 nights, a classic tourist product), **weekly** (7–29) and **monthly+** (30+, closer to a mid- or long-term rental). The cut-offs are numbers of nights, not money, so the same thresholds are valid in every city. The 12 listings with a missing `minimum_nights` get no group.

**Result:** monthly+ listings make up **37.1% in Barcelona and 46.8% in Los Angeles** (short stay: 60.4% vs 49.7%; weekly: 2.5% vs 3.5%). The direction of the team's prediction holds: there are more 30+ listings in Los Angeles. The size is the surprise: in both cities more than a third of all listings cannot be booked as a short tourist stay at all.

**Interpretation:** a minimum of one month makes these listings **mid-term (seasonal) rentals** rather than long-term housing. Likely guests are students, interns, remote workers or people relocating to the city, but the data contains no information about guests, so this is a hypothesis. Such listings compete less with hotels and more with the ordinary rental market.

In [ ]:
# Group listings into three rental models by the minimum stay the host accepts
bins = [0, 6, 29, np.inf]  # (0, 6] = 1-6 nights, (6, 29] = 7-29 nights, (29, inf) = 30+ nights
labels = ["Short stay (1-6 nights)", "Weekly (7-29)", "Monthly+ (30+)"]
df["rental_model"] = pd.cut(df["minimum_nights"], bins=bins, labels=labels)
df["rental_model"].value_counts(dropna=False)

In [ ]:
# Share of listings per rental model and city (%), as a table and a stacked bar chart
model_share = pd.crosstab(df["city_name"], df["rental_model"], normalize="index").mul(100).round(1)

colors = ["#86b6ef", "#2a78d6", "#104281"]  # one blue hue: lighter = shorter stays, darker = longer stays
ax = model_share.plot(kind="barh", stacked=True, color=colors, figsize=(9, 2.8),
                      width=0.6, edgecolor="white", linewidth=2)
for container, text_color in zip(ax.containers, ["black", "white", "white"]):
    labels = [f"{v:.1f}%" if v >= 5 else "" for v in container.datavalues]  # too narrow to label below 5%
    ax.bar_label(container, labels=labels, label_type="center", color=text_color, fontsize=10)
ax.invert_yaxis()  # first city on top
ax.set_xlim(0, 100)
ax.set_xlabel("% of listings")
ax.set_ylabel("")
ax.set_title("Rental model by city (from minimum_nights)")
ax.legend(title="", ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(FIG_DIR / "study04_rental_model_share.png", dpi=300, bbox_inches="tight")
plt.show()

model_share

## 7. Barcelona: licence status by rental model

If the 31–32-night peak is caused by the tourist-licence rules, monthly+ listings in Barcelona should **not** carry a tourist licence. In Barcelona the `license` field holds the regional number (`HUTB-...` for a tourist flat), the national registration number, or an explicit exemption such as `Exempt - seasonal rental`. The field has a different format in every city, so this check is done for Barcelona only and is not part of the replication.

**Result:** the licence field splits cleanly by rental model. **77.3% of short-stay listings carry a tourist licence (HUTB), against only 1.4% of monthly+ listings.** In turn, 52.2% of monthly+ listings are explicitly marked *exempt: seasonal rental*, a label that never appears among short-stay listings. Another 30.5% of monthly+ listings have no licence entry at all, which is legal for a stay longer than 31 days. This supports the interpretation in section 5: in Barcelona the monthly+ market is the part of Airbnb that operates outside the tourist-licence system.

In [ ]:
# Barcelona only: what the licence field says, per rental model (% of listings in each rental model)
bcn = df[df["city"] == "barcelona"]
licence = bcn["license"].fillna("")

licence_status = pd.Series("other (hotel, hostel, ...)", index=bcn.index, name="licence_status")
licence_status[licence == ""] = "missing"
licence_status[licence.str.contains("HUTB")] = "tourist licence (HUTB)"
licence_status[licence.str.contains("seasonal rental", case=False)] = "exempt: seasonal rental"

pd.crosstab(licence_status, bcn["rental_model"], normalize="columns").mul(100).round(1)

## 8. Are monthly+ listings really rented for a month?

A host could set a 30+ minimum only on paper and still rent out short stays. Reviews give an upper bound: a listing with a minimum of N nights can host at most `365 // N` stays a year, and each stay produces at most one review. A monthly+ listing with more reviews in the last 12 months than that is **implausible** as a pure monthly rental.

**Result:** only **0.5% (Barcelona) and 1.1% (Los Angeles)** of monthly+ listings exceed the upper bound. The median monthly+ listing got 0 reviews in the last 12 months, against 13 and 9 for short-stay listings. So the 30+ minimum is in almost all cases real, not a cover for short stays. **Limitation:** `minimum_nights` is the value at scrape time. A host who switched from short stays to 30+ during the year also counts as implausible, so 0.5–1.1% is an upper bound for disguised short stays. Hosts who rent short stays off-platform cannot be detected at all.

In [ ]:
# A stay of at least N nights allows at most 365 // N stays a year, and each stay gives at most one review.
# A monthly+ listing with more reviews in the last 12 months than that cannot have been rented only for 30+ nights.
monthly = df[df["rental_model"] == "Monthly+ (30+)"]
implausible = monthly["number_of_reviews_ltm"] > 365 // monthly["minimum_nights"]
short = df[df["rental_model"] == "Short stay (1-6 nights)"]

pd.DataFrame({
    "monthly_listings": monthly.groupby("city_name").size(),
    "median_reviews_ltm_monthly": monthly.groupby("city_name")["number_of_reviews_ltm"].median(),
    "median_reviews_ltm_short_stay": short.groupby("city_name")["number_of_reviews_ltm"].median(),
    "implausible_pct": implausible.groupby(monthly["city_name"]).mean().mul(100).round(1),
})

## 9. Seasonal switching: short stays in summer, monthly rentals in winter?

The data is a snapshot from late June 2026, so `minimum_nights` shows the summer setting. A model often described in Barcelona is short stays in summer and monthly rentals for the rest of the year. If hosts plan this ahead in their calendar, it is visible in two columns that look 365 nights into the future: `minimum_minimum_nights` (the smallest minimum stay in the calendar) and `maximum_minimum_nights` (the largest). A listing **switches rental model** if its calendar contains both a short-stay minimum (≤ 6 nights) and a monthly+ minimum (≥ 30 nights).

**Team's prediction:** about 10% of Barcelona listings switch, and fewer in Los Angeles, where stays are longer anyway.

**Result:** the prediction was too high. Only **1.6% of Barcelona listings (246) and 1.5% of Los Angeles listings (654)** switch between a short-stay and a monthly+ minimum within the next year, so the two cities hardly differ. Changes of the minimum stay are common (42.7% of Barcelona listings and 20.9% of Los Angeles listings), but small: among listings whose minimum varies, the median largest minimum is only 4 nights, e.g. a longer minimum on weekends or in peak weeks. In Barcelona, no listing marked *exempt: seasonal rental* switches (0%), as expected: without a tourist licence a short stay is not legal. Switching is most common among listings with no licence entry (2.8%). In the calendar data, the rental model is therefore a stable choice of the host, and the summer snapshot is a fair picture of the rest of the year.

**Limitations:** the two columns show *that* the minimum changes within the next year, but not *when*. A host who changes the setting by hand in September, or moves the flat to another platform for the winter, is invisible in a single snapshot. Testing that needs an earlier (winter) snapshot of the same listings, matched by `id`.

In [ ]:
# A listing switches rental model if its calendar for the next 365 nights contains both
# a short-stay minimum (<= 6 nights) and a monthly+ minimum (>= 30 nights)
df["varies_min_nights"] = df["minimum_minimum_nights"] != df["maximum_minimum_nights"]
df["switches_model"] = (df["minimum_minimum_nights"] <= 6) & (df["maximum_minimum_nights"] >= 30)

pd.DataFrame({
    "varies_pct": df.groupby("city_name")["varies_min_nights"].mean().mul(100).round(1),
    "switches_pct": df.groupby("city_name")["switches_model"].mean().mul(100).round(1),
    "switches_n": df.groupby("city_name")["switches_model"].sum(),
    # how big is the change for listings whose minimum varies at all?
    "median_largest_min_if_varies": df[df["varies_min_nights"]].groupby("city_name")["maximum_minimum_nights"].median(),
})

In [ ]:
# Barcelona: share of listings that switch model, per licence status (licence_status comes from section 7)
switches_bcn = df.loc[bcn.index, "switches_model"]
pd.DataFrame({
    "listings": switches_bcn.groupby(licence_status).size(),
    "switches_pct": switches_bcn.groupby(licence_status).mean().mul(100).round(1),
    "switches_n": switches_bcn.groupby(licence_status).sum(),
})

## Findings
_to fill in_